# SmartPantryAI — YOLO Training
**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Check GPU
import torch
print(f'torch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime!"}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → T4 GPU'

In [ ]:
# 2. Install ultralytics (torch already GPU-compatible on Colab)
!pip install ultralytics -q

In [ ]:
# 3. Mount Google Drive to persist best.pt
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/SmartPantryAI/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# 4. Download dataset from Kaggle
# Option A: if you have kaggle.json set up
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/SmartPantryAI/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d saikrishnamulukutla/food-ingredient-yolo-dataset --unzip -p /content/dataset

# Option B: upload dataset.zip manually via Files panel, then:
# !unzip -q /content/dataset.zip -d /content/dataset

# Option C: from Google Drive (if you saved it there)
# !unzip -q '/content/drive/MyDrive/SmartPantryAI/food_ingredient_dataset.zip' -d /content/dataset

import os
DATASET_ROOT = '/content/dataset'
print('Dataset files:', sum(1 for _ in __import__('pathlib').Path(DATASET_ROOT).rglob('*')))

In [ ]:
# 5. Patch data.yaml to point to Colab paths
import yaml, glob
yaml_files = glob.glob(f'{DATASET_ROOT}/**/dataset.yaml', recursive=True)
assert yaml_files, 'dataset.yaml not found — check DATASET_ROOT'
with open(yaml_files[0]) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(__import__('pathlib').Path(yaml_files[0]).parent)
with open('/content/data.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('data.yaml:')
print(open('/content/data.yaml').read())

In [ ]:
# 6. Train
!yolo detect train \
  data=/content/data.yaml \
  model=yolo11m.pt \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  name=food_yolo11 \
  project=/content/runs \
  exist_ok=True

In [ ]:
# 7. Save best.pt to Google Drive
import shutil, glob
best = glob.glob('/content/runs/**/best.pt', recursive=True)
if best:
    dest = f'{OUTPUT_DIR}/best.pt'
    shutil.copy(best[0], dest)
    size = __import__('os').path.getsize(dest) / 1e6
    print(f'Saved to Drive: {dest}  ({size:.1f} MB)')
else:
    print('ERROR: best.pt not found')

In [ ]:
# 8. Print final metrics
import pandas as pd, glob
results = glob.glob('/content/runs/**/results.csv', recursive=True)
if results:
    df = pd.read_csv(results[0])
    df.columns = df.columns.str.strip()
    cols = [c for c in ['epoch','metrics/mAP50(B)','metrics/mAP50-95(B)'] if c in df.columns]
    print(df[cols].tail(10).to_string(index=False))